# Step 3A — Materialize all-cell ResolVI-corrected expression as AnnData Zarr

Run this notebook in the **same scvi-tools / ResolVI environment** used for
Step 2.

For every successfully trained sample it creates one authoritative all-cell
Zarr store:

```python
adata.X
# sparse integer Proseg counts

adata.layers["resolvi_corrected_10k"]
# dense float32 ResolVI posterior expected true expression,
# scaled to a common library size of 10,000
```

The dense corrected layer is decoded in cell chunks into a disk-backed NumPy
memmap and then written to chunked Zarr. The full corrected matrix is never
assembled in RAM. The raw matrix remains in `X`, so count-based analyses cannot
accidentally use corrected floating-point expression.

This notebook does **not** run pyUCell or reference annotation. Those are kept
separate so the corrected matrix is generated once and reused consistently.


In [1]:
# ---------------------------------------------------------------------
# Environment — run before importing torch, scvi-tools, or ResolVI
# ---------------------------------------------------------------------
import os

GPU_ID = "0"  # use a different physical GPU in each parallel kernel
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONHASHSEED"] = "0"
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"

print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


CUDA_VISIBLE_DEVICES: 0


In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import shutil
import time
import warnings
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import pyro
import scipy.sparse as sp
import torch
import zarr

import scvi
from scvi.external import RESOLVI

SAMPLE_INFO = {
    "Screen_39_21": {"patient": "patient_39_21", "cancer_type": "NSCLC", "biopsy_stage": "Screen"},
    "C2D15_39_21": {"patient": "patient_39_21", "cancer_type": "NSCLC", "biopsy_stage": "C2D15"},
    "Screen_17_26": {"patient": "patient_17_26", "cancer_type": "NSCLC", "biopsy_stage": "Screen"},
    "C2D15_17_26": {"patient": "patient_17_26", "cancer_type": "NSCLC", "biopsy_stage": "C2D15"},
    "Screen_18_23": {"patient": "patient_18_23", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_18_23": {"patient": "patient_18_23", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_16_22": {"patient": "patient_16_22", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_16_22": {"patient": "patient_16_22", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_30_16": {"patient": "patient_30_16", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_30_16": {"patient": "patient_30_16", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_23_25": {"patient": "patient_23_25", "cancer_type": "colon_cancer", "biopsy_stage": "Screen"},
    "C2D15_23_25": {"patient": "patient_23_25", "cancer_type": "colon_cancer", "biopsy_stage": "C2D15"},
}

print("scvi-tools:", scvi.__version__)
print("torch:", str(torch.__version__))
print("CUDA available:", torch.cuda.is_available())
print("Visible CUDA devices:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for ResolVI decoding.")

torch.set_float32_matmul_precision("high")
scvi.settings.seed = 0


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 0


scvi-tools: 1.5.0.post1
torch: 2.11.0+cu130
CUDA available: True
Visible CUDA devices: 1


In [3]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path("/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057")
PIPELINE_ROOT = PROJECT_ROOT / "tmp" / "proseg_resolvi_immune_enrichment_v1"
RESOLVI_ROOT = PIPELINE_ROOT / "02_resolvi"
ALLCELL_ZARR_ROOT = PIPELINE_ROOT / "03a_resolvi_allcell_zarr"
STAGING_ROOT = PIPELINE_ROOT / "03a_resolvi_allcell_zarr_staging"
ALLCELL_ZARR_ROOT.mkdir(parents=True, exist_ok=True)
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

INCLUDE_COLON = True
SECTION_NAMES = [
    sample
    for sample, meta in SAMPLE_INFO.items()
    if INCLUDE_COLON or meta["cancer_type"] != "colon_cancer"
]

# For a quick integration test, set SECTION_NAMES to one completed sample.
# SECTION_NAMES = ["C2D15_23_25"]

SCVI_ACCELERATOR = "gpu"
SCVI_DEVICE_SPEC = 1  # one GPU visible through CUDA_VISIBLE_DEVICES
NORMALIZED_LIBRARY_SIZE = 10_000.0
DECODE_CELL_CHUNK = 1_000
POSTERIOR_BATCH_SIZE = 512
CORRECTED_DTYPE = np.float32
CORRECTED_LAYER = "resolvi_corrected_10k"

# Chunk shape used by AnnData.write_zarr for dense arrays.
ZARR_CHUNKS = (512, 2_048)

REUSE_COMPLETED_ZARR = True
OVERWRITE_ZARR = False
RESUME_MEMMAP = True
DELETE_STAGING_MEMMAP_AFTER_SUCCESS = True
CONTINUE_ON_ERROR = True

# The temporary memmap and the final Zarr coexist during export.
# Require a conservative amount of free space before decoding.
MIN_DISK_MULTIPLIER_OVER_DENSE = 2.2
MIN_EXTRA_FREE_GIB = 10.0

PIPELINE_VERSION = "2026-07-29-step3a-allcell-zarr-v1"

print("Samples:", SECTION_NAMES)
print("All-cell Zarr root:", ALLCELL_ZARR_ROOT)


Samples: ['Screen_39_21', 'C2D15_39_21', 'Screen_17_26', 'C2D15_17_26', 'Screen_18_23', 'C2D15_18_23', 'Screen_16_22', 'C2D15_16_22', 'Screen_30_16', 'C2D15_30_16', 'Screen_23_25', 'C2D15_23_25']
All-cell Zarr root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03a_resolvi_allcell_zarr


In [4]:
# ---------------------------------------------------------------------
# Paths and validation helpers
# ---------------------------------------------------------------------
def paths_for_sample(sample: str) -> dict[str, Path]:
    resolvi_dir = RESOLVI_ROOT / sample
    output_dir = ALLCELL_ZARR_ROOT / sample
    staging_dir = STAGING_ROOT / sample
    output_dir.mkdir(parents=True, exist_ok=True)
    staging_dir.mkdir(parents=True, exist_ok=True)
    return {
        "sample": Path(sample),
        "prepared": resolvi_dir / f"{sample}_resolvi_prepared.h5ad",
        "final_h5ad": resolvi_dir / f"{sample}_resolvi_annotated.h5ad",
        "model": resolvi_dir / "model",
        "zarr": output_dir / f"{sample}_resolvi_allcells.zarr",
        "summary": output_dir / f"{sample}_resolvi_allcells_zarr_summary.json",
        "memmap": staging_dir / f"{sample}_{CORRECTED_LAYER}.float32.dat",
        "progress": staging_dir / f"{sample}_{CORRECTED_LAYER}_progress.json",
    }


def write_json(payload, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temp.replace(path)


def names_hash(names) -> str:
    digest = hashlib.sha256()
    for value in names.astype(str):
        digest.update(value.encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def complete_model_checkpoint_exists(model_dir: Path) -> bool:
    return model_dir.is_dir() and (model_dir / "model.pt").is_file()


def exact_dense_bytes(n_obs: int, n_vars: int, dtype=np.float32) -> int:
    return int(n_obs) * int(n_vars) * np.dtype(dtype).itemsize


def available_disk_bytes(path: Path) -> int:
    return shutil.disk_usage(path).free


def close_memmap(array) -> None:
    try:
        array.flush()
    except Exception:
        pass
    mmap_obj = getattr(array, "_mmap", None)
    if mmap_obj is not None:
        try:
            mmap_obj.close()
        except Exception:
            pass


def validate_alignment(prepared: ad.AnnData, final: ad.AnnData) -> None:
    if prepared.shape != final.shape:
        raise ValueError(
            f"Prepared/final shape mismatch: {prepared.shape} versus {final.shape}."
        )
    if not prepared.obs_names.equals(final.obs_names):
        raise ValueError("Prepared and final obs_names are not identically ordered.")
    if not prepared.var_names.equals(final.var_names):
        raise ValueError("Prepared and final var_names are not identically ordered.")
    if not sp.issparse(final.X):
        final.X = sp.csr_matrix(final.X)
    final.X = sp.csr_matrix(final.X).astype(np.int32)


def validate_existing_zarr(paths: dict[str, Path], expected_shape) -> bool:
    if not paths["zarr"].exists() or not paths["summary"].exists():
        return False
    try:
        root = zarr.open_group(str(paths["zarr"]), mode="r")
        layer = root["layers"][CORRECTED_LAYER]
        if tuple(layer.shape) != tuple(expected_shape):
            return False
        summary = json.loads(paths["summary"].read_text(encoding="utf-8"))
        return (
            summary.get("pipeline_version") == PIPELINE_VERSION
            and summary.get("completed") is True
            and tuple(summary.get("shape", [])) == tuple(expected_shape)
        )
    except Exception:
        return False


In [5]:
# ---------------------------------------------------------------------
# Chunked decoding and Zarr export
# ---------------------------------------------------------------------
def decode_corrected_to_memmap(
    model,
    prepared: ad.AnnData,
    paths: dict[str, Path],
) -> tuple[np.memmap, dict]:
    shape = prepared.shape
    expected_bytes = exact_dense_bytes(*shape, dtype=CORRECTED_DTYPE)
    progress_payload = None

    if paths["progress"].exists() and paths["memmap"].exists() and RESUME_MEMMAP:
        progress_payload = json.loads(paths["progress"].read_text(encoding="utf-8"))
        compatible = (
            tuple(progress_payload.get("shape", [])) == tuple(shape)
            and progress_payload.get("dtype") == np.dtype(CORRECTED_DTYPE).name
            and float(progress_payload.get("library_size")) == float(NORMALIZED_LIBRARY_SIZE)
            and paths["memmap"].stat().st_size == expected_bytes
        )
        if not compatible:
            print("Discarding incompatible staging memmap.")
            paths["memmap"].unlink(missing_ok=True)
            paths["progress"].unlink(missing_ok=True)
            progress_payload = None

    if progress_payload is None:
        memmap = np.memmap(
            paths["memmap"],
            mode="w+",
            dtype=CORRECTED_DTYPE,
            shape=shape,
        )
        next_row = 0
        nonzero_entries = 0
        total_entries_seen = 0
        min_value = None
        max_value = None
        progress_payload = {
            "pipeline_version": PIPELINE_VERSION,
            "shape": list(map(int, shape)),
            "dtype": np.dtype(CORRECTED_DTYPE).name,
            "library_size": float(NORMALIZED_LIBRARY_SIZE),
            "next_row": 0,
            "nonzero_entries": 0,
            "total_entries_seen": 0,
            "min_value": None,
            "max_value": None,
            "completed": False,
        }
        write_json(progress_payload, paths["progress"])
    else:
        memmap = np.memmap(
            paths["memmap"],
            mode="r+",
            dtype=CORRECTED_DTYPE,
            shape=shape,
        )
        next_row = int(progress_payload.get("next_row", 0))
        nonzero_entries = int(progress_payload.get("nonzero_entries", 0))
        total_entries_seen = int(progress_payload.get("total_entries_seen", 0))
        min_value = progress_payload.get("min_value")
        max_value = progress_payload.get("max_value")
        print(f"Resuming corrected-expression decoding at row {next_row:,}.")

    for start in range(next_row, prepared.n_obs, int(DECODE_CELL_CHUNK)):
        end = min(start + int(DECODE_CELL_CHUNK), prepared.n_obs)
        indices = np.arange(start, end, dtype=np.int64)

        try:
            chunk = model.get_normalized_expression(
                adata=prepared,
                indices=indices,
                gene_list=None,
                library_size=float(NORMALIZED_LIBRARY_SIZE),
                n_samples=1,
                return_mean=True,
                return_numpy=True,
                batch_size=int(POSTERIOR_BATCH_SIZE),
            )
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            warnings.warn(
                "CUDA OOM during corrected decoding; retrying this chunk with "
                "posterior batch size 128."
            )
            torch.cuda.empty_cache()
            chunk = model.get_normalized_expression(
                adata=prepared,
                indices=indices,
                gene_list=None,
                library_size=float(NORMALIZED_LIBRARY_SIZE),
                n_samples=1,
                return_mean=True,
                return_numpy=True,
                batch_size=128,
            )

        chunk = np.asarray(chunk, dtype=CORRECTED_DTYPE)
        expected_chunk_shape = (end - start, prepared.n_vars)
        if chunk.shape != expected_chunk_shape:
            raise ValueError(
                f"Corrected chunk shape {chunk.shape}; expected {expected_chunk_shape}."
            )
        if not np.isfinite(chunk).all():
            raise FloatingPointError("Corrected expression contains non-finite values.")

        memmap[start:end, :] = chunk
        memmap.flush()

        nonzero_entries += int(np.count_nonzero(chunk))
        total_entries_seen += int(chunk.size)
        chunk_min = float(chunk.min())
        chunk_max = float(chunk.max())
        min_value = chunk_min if min_value is None else min(float(min_value), chunk_min)
        max_value = chunk_max if max_value is None else max(float(max_value), chunk_max)

        progress_payload.update(
            {
                "next_row": int(end),
                "nonzero_entries": int(nonzero_entries),
                "total_entries_seen": int(total_entries_seen),
                "min_value": float(min_value),
                "max_value": float(max_value),
                "completed": bool(end == prepared.n_obs),
            }
        )
        write_json(progress_payload, paths["progress"])
        print(
            f"Decoded {end:,}/{prepared.n_obs:,} cells "
            f"({100.0 * end / prepared.n_obs:.1f}%)."
        )
        del chunk
        gc.collect()
        torch.cuda.empty_cache()

    return memmap, progress_payload


def export_sample_allcell_zarr(sample: str) -> dict:
    paths = paths_for_sample(sample)
    print("\n" + "=" * 90)
    print("All-cell corrected Zarr sample:", sample)

    for key in ("prepared", "final_h5ad"):
        if not paths[key].exists():
            raise FileNotFoundError(paths[key])
    if not complete_model_checkpoint_exists(paths["model"]):
        raise FileNotFoundError(paths["model"] / "model.pt")

    final = ad.read_h5ad(paths["final_h5ad"])
    prepared = ad.read_h5ad(paths["prepared"])
    validate_alignment(prepared, final)

    if REUSE_COMPLETED_ZARR and not OVERWRITE_ZARR and validate_existing_zarr(
        paths, final.shape
    ):
        print("Reusing existing completed all-cell Zarr:", paths["zarr"])
        return json.loads(paths["summary"].read_text(encoding="utf-8"))

    dense_bytes = exact_dense_bytes(*final.shape, dtype=CORRECTED_DTYPE)
    final_h5ad_bytes = paths["final_h5ad"].stat().st_size
    required = int(
        dense_bytes * float(MIN_DISK_MULTIPLIER_OVER_DENSE)
        + final_h5ad_bytes
        + float(MIN_EXTRA_FREE_GIB) * 2**30
    )
    free = available_disk_bytes(ALLCELL_ZARR_ROOT)
    print(f"Corrected dense matrix estimate: {dense_bytes / 2**30:.1f} GiB")
    print(f"Available disk: {free / 2**30:.1f} GiB")
    print(f"Conservative required free disk: {required / 2**30:.1f} GiB")
    if free < required:
        raise OSError(
            "Insufficient free disk for the staging memmap plus final Zarr. "
            f"Required about {required / 2**30:.1f} GiB, available {free / 2**30:.1f} GiB."
        )

    pyro.clear_param_store()
    model = RESOLVI.load(
        str(paths["model"]),
        adata=prepared,
        accelerator=SCVI_ACCELERATOR,
        device=SCVI_DEVICE_SPEC,
    )

    started = time.time()
    memmap, progress = decode_corrected_to_memmap(model, prepared, paths)

    final.layers[CORRECTED_LAYER] = memmap
    final.uns["resolvi_corrected_expression"] = {
        "pipeline_version": PIPELINE_VERSION,
        "layer": CORRECTED_LAYER,
        "raw_count_location": "X",
        "library_size": float(NORMALIZED_LIBRARY_SIZE),
        "dtype": np.dtype(CORRECTED_DTYPE).name,
        "interpretation": (
            "ResolVI posterior expected true normalized expression; "
            "not observed integer counts"
        ),
        "decode_cell_chunk": int(DECODE_CELL_CHUNK),
        "posterior_batch_size": int(POSTERIOR_BATCH_SIZE),
    }

    temp_zarr = paths["zarr"].with_name(paths["zarr"].name + ".tmp")
    if temp_zarr.exists():
        shutil.rmtree(temp_zarr)
    if paths["zarr"].exists():
        if not OVERWRITE_ZARR:
            raise FileExistsError(paths["zarr"])
        shutil.rmtree(paths["zarr"])

    print("Writing AnnData Zarr ...")
    final.write_zarr(str(temp_zarr), chunks=ZARR_CHUNKS)
    shutil.move(str(temp_zarr), str(paths["zarr"]))

    root = zarr.open_group(str(paths["zarr"]), mode="r")
    corrected_disk = root["layers"][CORRECTED_LAYER]
    if tuple(corrected_disk.shape) != tuple(final.shape):
        raise ValueError("Written corrected Zarr layer has the wrong shape.")

    obs_hash = names_hash(final.obs_names)
    var_hash = names_hash(final.var_names)
    density = (
        float(progress["nonzero_entries"]) / float(progress["total_entries_seen"])
        if progress["total_entries_seen"]
        else float("nan")
    )
    summary = {
        "pipeline_version": PIPELINE_VERSION,
        "sample": sample,
        **SAMPLE_INFO[sample],
        "completed": True,
        "zarr_path": str(paths["zarr"]),
        "shape": list(map(int, final.shape)),
        "raw_count_location": "X",
        "corrected_layer": CORRECTED_LAYER,
        "corrected_dtype": str(corrected_disk.dtype),
        "corrected_chunks": list(map(int, corrected_disk.chunks)),
        "corrected_exact_density": density,
        "corrected_min": progress.get("min_value"),
        "corrected_max": progress.get("max_value"),
        "library_size": float(NORMALIZED_LIBRARY_SIZE),
        "obs_names_sha256": obs_hash,
        "var_names_sha256": var_hash,
        "runtime_minutes": (time.time() - started) / 60.0,
    }
    write_json(summary, paths["summary"])
    print("Saved:", paths["zarr"])
    print("Corrected exact density:", f"{density:.4%}")

    # Release the memmap before deleting the staging file.
    del final.layers[CORRECTED_LAYER]
    close_memmap(memmap)
    del memmap, model, prepared, final
    pyro.clear_param_store()
    gc.collect()
    torch.cuda.empty_cache()

    if DELETE_STAGING_MEMMAP_AFTER_SUCCESS:
        paths["memmap"].unlink(missing_ok=True)
        paths["progress"].unlink(missing_ok=True)

    return summary


## Run selected samples

For four-way parallelization, open four kernels with distinct `GPU_ID` values
and disjoint `SECTION_NAMES`. Avoid writing the same sample from two kernels.


In [6]:
allcell_results = {}
allcell_failures = {}

for sample in SECTION_NAMES:
    try:
        allcell_results[sample] = export_sample_allcell_zarr(sample)
    except Exception as exc:
        allcell_failures[sample] = repr(exc)
        print(f"[FAILED] {sample}: {type(exc).__name__}: {exc}")
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        pyro.clear_param_store()
        gc.collect()
        torch.cuda.empty_cache()

pd.DataFrame.from_dict(allcell_results, orient="index").to_csv(
    ALLCELL_ZARR_ROOT / "all_samples_allcell_zarr_summary.csv"
)
write_json(
    allcell_failures,
    ALLCELL_ZARR_ROOT / "all_samples_allcell_zarr_failures.json",
)

print("Completed:", sorted(allcell_results))
print("Failures:", json.dumps(allcell_failures, indent=2))



All-cell corrected Zarr sample: Screen_39_21
Corrected dense matrix estimate: 1.5 GiB
Available disk: 8589409857.6 GiB
Conservative required free disk: 13.4 GiB
INFO     File                                                                                                      
         /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/02_re
         solvi/Screen_39_21/model/model.pt already downloaded                                                      


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 0 in adata.obs['_scvi_ind_x'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/scvi/model/base/_save_load.py:161: FutureWarning: RESOLVI is a spatial transcriptomics model that will be moved to the scvi-tools spatial companion package `scviva-tools` starting in scvi-tools v1.5 and will no longer be supported here. It will be deprecated from scvi-tools in v1.6.
  model = cls(adata, **non_kwargs, **kwargs)
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/scvi/data/_utils.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violate

Decoded 1,000/22,950 cells (4.4%).
Decoded 2,000/22,950 cells (8.7%).
Decoded 3,000/22,950 cells (13.1%).
Decoded 4,000/22,950 cells (17.4%).
Decoded 5,000/22,950 cells (21.8%).
Decoded 6,000/22,950 cells (26.1%).
Decoded 7,000/22,950 cells (30.5%).
Decoded 8,000/22,950 cells (34.9%).
Decoded 9,000/22,950 cells (39.2%).
Decoded 10,000/22,950 cells (43.6%).
Decoded 11,000/22,950 cells (47.9%).
Decoded 12,000/22,950 cells (52.3%).
Decoded 13,000/22,950 cells (56.6%).
Decoded 14,000/22,950 cells (61.0%).
Decoded 15,000/22,950 cells (65.4%).
Decoded 16,000/22,950 cells (69.7%).
Decoded 17,000/22,950 cells (74.1%).
Decoded 18,000/22,950 cells (78.4%).
Decoded 19,000/22,950 cells (82.8%).
Decoded 20,000/22,950 cells (87.1%).
Decoded 21,000/22,950 cells (91.5%).
Decoded 22,000/22,950 cells (95.9%).
Decoded 22,950/22,950 cells (100.0%).
Writing AnnData Zarr ...


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


[FAILED] Screen_39_21: IORegistryError: No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>

All-cell corrected Zarr sample: C2D15_39_21
Corrected dense matrix estimate: 0.7 GiB
Available disk: 8589409857.6 GiB
Conservative required free disk: 11.6 GiB
INFO     File                                                                                                      
         /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/02_re
         solvi/C2D15_39_21/model/model.pt already downloaded                                                       


...

[FAILED] Screen_23_25: IORegistryError: No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>

All-cell corrected Zarr sample: C2D15_23_25
Corrected dense matrix estimate: 0.6 GiB
Available disk: 8589409830.7 GiB
Conservative required free disk: 11.3 GiB
INFO     File                                                                                                      
         /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/02_re
         solvi/C2D15_23_25/model/model.pt already downloaded                                                       


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 0 in adata.obs['_scvi_ind_x'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/scvi/model/base/_save_load.py:161: FutureWarning: RESOLVI is a spatial transcriptomics model that will be moved to the scvi-tools spatial companion package `scviva-tools` starting in scvi-tools v1.5 and will no longer be supported here. It will be deprecated from scvi-tools in v1.6.
  model = cls(adata, **non_kwargs, **kwargs)


Decoded 1,000/10,243 cells (9.8%).
Decoded 2,000/10,243 cells (19.5%).
Decoded 3,000/10,243 cells (29.3%).
Decoded 4,000/10,243 cells (39.1%).
Decoded 5,000/10,243 cells (48.8%).
Decoded 6,000/10,243 cells (58.6%).
Decoded 7,000/10,243 cells (68.3%).
Decoded 8,000/10,243 cells (78.1%).
Decoded 9,000/10,243 cells (87.9%).
Decoded 10,000/10,243 cells (97.6%).
Decoded 10,243/10,243 cells (100.0%).
Writing AnnData Zarr ...


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


[FAILED] C2D15_23_25: IORegistryError: No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>
Completed: []
Failures: {
  "Screen_39_21": "IORegistryError(\"No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>\")",
  "C2D15_39_21": "IORegistryError(\"No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>\")",
  "Screen_17_26": "IORegistryError(\"No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>\")",
  "C2D15_17_26": "IORegistryError(\"No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>\")",
  "Screen_18_23": "IORegistryError(\"No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>\")",
  "C2D15_18_23": "IORegistryError(\"No method registered for writing <class 'numpy.memmap'> into <class 'zarr.core.group.Group'>\")",
  "Screen_16_22": "IORegistryError(

In [7]:
#also record environment location
import sys
print(sys.executable)

/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python


## Lazy downstream access example

```python
import anndata as ad

lazy = ad.experimental.read_lazy(
    "/path/to/Screen_30_16_resolvi_allcells.zarr"
)

lazy.X
# lazy sparse raw counts

lazy.layers["resolvi_corrected_10k"]
# lazy dense corrected expression
```
